In [ ]:
%pip install "rheofit>=1.0.1"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rheofit

print("rheofit", rheofit.__version__)

# rheofit intro — fit a flow curve with the TC–Carreau model

This notebook shows the basic rheofit workflow on real data: a **0.25 % Carbopol**
sample measured in a TRIOS rheometer, exported as an Excel workbook
(`carbopol_0_25.xls`). The workbook contains two worksheets:

1. **Flow sweep - 1** — shear-rate sweep → stress (the flow curve we fit)
2. **Amplitude sweep - 2** — oscillatory strain sweep → moduli (shown at the end)

Since rheofit 1.0 the library works on plain DataFrames with canonical column
names (`Shear rate / 1/s`, `Stress / Pa`, …), so we read the worksheets with
pandas and go straight to fitting.

In [ ]:
def load_flow_sheet(path, sheet):
    """Read one TRIOS worksheet into a canonical flow-curve DataFrame."""
    raw = pd.read_excel(path, sheet_name=sheet, header=1)
    df = pd.DataFrame({
        "Shear rate / 1/s": pd.to_numeric(raw["Shear rate"], errors="coerce"),
        "Stress / Pa": pd.to_numeric(raw["Stress"], errors="coerce"),
    }).dropna()
    df = df[(df["Shear rate / 1/s"] > 0) & (df["Stress / Pa"] > 0)]
    return df.reset_index(drop=True)

df = load_flow_sheet("carbopol_0_25.xls", "Flow sweep - 1")
print(f"{len(df)} points")
df.head()

In [ ]:
rheofit.plot(df, test_type="flow_curve")

## Define model: TC–Carreau (TCC)

We describe the sample with a **TCC model**: the **TC model** for the yield /
plastic contribution of the crosslinked Carbopol microgel particles, plus a
**Carreau term** for the shear-thinning continuous phase.

### Carreau term

$$ \sigma=\dot\gamma \cdot \eta_0 \cdot (1+(\lambda\dot\gamma)^2)^{-1/2} $$

### TC term

$$ \sigma = \sigma_y + \sigma_y (\dot\gamma / \dot\gamma_c)^{0.5} + \eta_{bg} \dot\gamma $$

### TCC model

$$ \sigma = \sigma_y + \sigma_y (\dot\gamma / \dot\gamma_c)^{0.5} + \dot\gamma \cdot \eta_0 \cdot (1+(\lambda\dot\gamma)^2)^{-1/2} $$

In rheofit ≥ 1.0 this combination ships as the built-in model **`tc_carreau`**
(the old `models.TC_model + models.carreau_model` composition with
`TC_eta_bg` fixed to 0 is no longer needed):

In [ ]:
rheofit.model_info("tc_carreau")

In [ ]:
res = rheofit.fit(df, "tc_carreau")

In [ ]:
params = pd.DataFrame.from_dict(res["params"], orient="index")
params.index.name = "parameter"
display(params)
print(f"\nRedChi2 = {res['redchi']:.3e}   (dimensionless, relative residuals)")

In [ ]:
rheofit.plot(df, fits=res, test_type="flow_curve")

## The amplitude sweep

The second worksheet is an oscillatory **amplitude sweep** (strain sweep at
fixed frequency). Plotting the storage and loss moduli shows the linear
viscoelastic plateau and the yielding of the microgel network:

In [ ]:
raw = pd.read_excel("carbopol_0_25.xls", sheet_name="Amplitude sweep - 2", header=1)
osc = pd.DataFrame({
    "Oscillation strain / %": pd.to_numeric(raw["Oscillation strain"], errors="coerce"),
    "Storage modulus / Pa": pd.to_numeric(raw["Storage modulus"], errors="coerce"),
    "Loss modulus / Pa": pd.to_numeric(raw["Loss modulus"], errors="coerce"),
    "Oscillation stress / Pa": pd.to_numeric(raw["Oscillation stress"], errors="coerce"),
}).dropna().reset_index(drop=True)

rheofit.plot(osc, test_type="amplitude_sweep")

In [ ]:
plt.figure()
plt.loglog(osc["Oscillation strain / %"], osc["Oscillation stress / Pa"],
           ".", color="black", mfc="none", markersize=10, label="oscillation stress")
plt.xlabel("Oscillation strain [%]")
plt.ylabel("Stress [Pa]")
plt.xlim(0.01, 10000)
plt.ylim(0.1)
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()